In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from scipy.ndimage import gaussian_filter

conic_dataset = np.load("dataset/dataset_CONIC.npz")
nadir_dataset = np.load("dataset/dataset_NADIR.npz")
pointwise_dataset = np.load("dataset/dataset_POINTWISE.npz")

In [ ]:
print("Conic Dataset:")
print(conic_dataset.files)
print("X shape:", conic_dataset["observed_mask"].shape)

In [ ]:
def plot_dataset(dataset, title):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(dataset['observed_map'][0], cmap='gray')
    axes[0].set_title('Input Image')
    axes[1].imshow(dataset['ground_truth'][0], cmap='gray')
    axes[1].set_title('Ground Truth')
    axes[2].imshow(dataset['observed_mask'][0], cmap='gray')
    axes[2].set_title('Mask')
    
    for ax in axes:
        ax.axis('off')
    
    plt.suptitle(title)
    plt.show()


In [ ]:
plot_dataset(conic_dataset, "Conic Dataset")

In [ ]:
def plot_sensor_comparison(
    conic_dataset,
    nadir_dataset,
    pointwise_dataset,
    sample_index=0,
):
    datasets = {
        "CONIC": conic_dataset,
        "NADIR": nadir_dataset,
        "POINTWISE": pointwise_dataset,
    }

    ground_truth = conic_dataset["ground_truth"][sample_index]

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.ravel()

    vmin = ground_truth.min()
    vmax = ground_truth.max()

    # Ground truth sin observaciones
    axes[0].imshow(
        ground_truth,
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
        interpolation="bicubic",
    )
    axes[0].set_title("Ground Truth")
    axes[0].set_xlabel("X")
    axes[0].set_ylabel("Y")

    # Cada modelo de observación
    for ax, (sensor_name, dataset) in zip(axes[1:], datasets.items()):
        current_ground_truth = dataset["ground_truth"][sample_index]
        observed_mask = dataset["observed_mask"][sample_index]

        ax.imshow(
            current_ground_truth,
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
            interpolation="bicubic",
        )

        # Zona observada con transparencia
        ax.imshow(
            np.ma.masked_where(observed_mask == 0, observed_mask),
            cmap="Reds",
            vmin=0,
            vmax=1,
            alpha=0.35,
            interpolation="nearest",
        )

        # Borde de la zona observada, sin modificar la máscara
        ax.contour(
            observed_mask,
            levels=[0.5],
            colors="white",
            linewidths=1.2,
            antialiased=True,
        )

        ax.set_title(f"{sensor_name}: Ground Truth + Observations")
        ax.set_xlabel("X")
        ax.set_ylabel("Y")

    for ax in axes:
        ax.set_aspect("equal")
        ax.grid(False)

    fig.suptitle(
        f"Comparison of Observation Models - Sample {sample_index}",
        fontsize=16,
    )

    fig.tight_layout()
    plt.show()

    return fig, axes

In [ ]:
plot_sensor_comparison(
    conic_dataset,
    nadir_dataset,
    pointwise_dataset,
    sample_index=0,
)